In [ ]:
import pandas as pd

from srm.analysis import BCSDRun, load_nasa_nex
from srm.bcsd_config import BCSDConfig, VariableConfig

In [ ]:
lat, lon = (-26.2056, 28.0337)  # johannesburg

## get CarbonPlan data
Easiest way to do that is to go through our config system, which you can pass to BCSDRun. 
That will allow you load historical and scenario data without having to look anything up on S3

In [ ]:
vc = VariableConfig.for_variable("tas").model_copy(update={"do_windowing": False})
bcsd_config = BCSDConfig(
    gcm="CESM2-WACCM",
    variable="tas",
    ensemble_member="r1i1p1f1",
    subset_bounds=(-35, -22, 16, 33),
    scenario="ssp245",
    predict_period_start=2015,
    predict_period_end=2100,
    variable_config=vc,
    output_dir="s3://carbonplan-scratch/srm/outputs/window",
    cache_dir="s3://carbonplan-scratch/srm/bcsd_cache/window",
)
r = BCSDRun(bcsd_config)

cp_pt = r.historical["tas"].sel(lat=lat, lon=lon, method="nearest")
cp_df = cp_pt.to_dataframe()

## Get NASA-NEX data
again, we have a little helper function.
you can load two datasets: `historical` or `ssp245`. 
NASA data is (currently?) only for the `r3...` ensemble member so keep that in mind when doing your comparisons. 
We haven't run that ensemble member using our approach yet so the below is just a demo. 

N.B. Getting NASA NEX data is pretty slow for now because of how it's chunked. If you find yourself doing a ton of these comparisons, we should either i) rechunk the data or ii) grab multiple points of interest in one go. 

So something like: 

```
point_pairs = [(47,-110),(36, -65), (5,5)]
lats, lons = zip(*point_pairs)

point_coords = {
    "lat": xr.DataArray(list(lats), dims="points"),
    "lon": xr.DataArray(list(lons), dims="points"),
}

result = ds['tas'].sel(point_coords, method="nearest")
```

would have to fix the join at the end as well.

In [ ]:
nnh = load_nasa_nex(dataset="historical")
nnh_pt = nnh["tas"].sel(lat=lat, lon=lon, method="nearest")
nnh_df = nnh_pt.to_dataframe()  # this is painfully slow since we have pancake chunking right now
nnh_df = nnh_df.droplevel(0)  # strip off ensemble_member index

In [ ]:
compare_data = pd.merge(
    cp_df[["tas"]], nnh_df[["tas"]], left_index=True, right_index=True, suffixes=["_cp", "_nasa"]
)
compare_data.plot()